In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [10]:
# shape: 4 * 3 * 128 * 128
class BatchNorm(nn.Module):
    def __init__(self, num_channel, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.gamma = nn.Parameter(torch.ones(num_channel))
        self.beta = nn.Parameter(torch.zeros(num_channel))
        self.register_buffer('running_mean', torch.zeros(num_channel))
        self.register_buffer('running_var', torch.ones(num_channel))
        

    def forward(self, x, training=True):
        '''
        x: (batch_size, num_channel, height, width)
        '''
        if self.training: # train()
            var = x.var(dim=(0, 2, 3), keepdim=True) # var.shape = (1, num_channel, 1, 1)
            mean = x.mean(dim=(0, 2, 3), keepdim=True) # mean.shape = (1, num_channel, 1, 1)
            self.running_mean = (1-self.momentum)*self.running_mean+self.momentum*mean
            self.running_var = (1-self.momentum)*self.running_var+self.momentum*var
        else: # eval()
            mean = self.running_mean
            var = self.running_var
    
        x = (x-mean)/torch.sqrt(var + self.eps)
        return self.gamma*x+self.beta

In [8]:
# Batch Norm Complete Version
import torch
import torch.nn as nn


class BatchNorm(nn.Module):
    """
    处理形状为 (N, C, H, W) 的四维输入。
    
    N: batch size
    C: channel 数量
    H: height
    W: width
    """

    def __init__(self, num_channel, eps=1e-5, momentum=0.1):
        super().__init__()

        self.num_channel = num_channel
        self.eps = eps
        self.momentum = momentum

        # 可学习参数：
        # gamma 控制缩放，beta 控制平移
        self.gamma = nn.Parameter(
            torch.ones(num_channel)
        )
        self.beta = nn.Parameter(
            torch.zeros(num_channel)
        )

        # 非可学习的历史统计量：
        # 会被模型保存，但不会被优化器更新
        self.register_buffer(
            "running_mean",
            torch.zeros(num_channel)
        )
        self.register_buffer(
            "running_var",
            torch.ones(num_channel)
        )

    def forward(self, x):
        # 检查输入是否为 (N, C, H, W)
        if x.ndim != 4:
            raise ValueError(
                f"输入必须是 (N, C, H, W)，"
                f"但现在的形状是 {tuple(x.shape)}"
            )

        # 检查输入 channel 数是否正确
        if x.shape[1] != self.num_channel:
            raise ValueError(
                f"BatchNorm 创建时指定了 "
                f"{self.num_channel} 个 channel，"
                f"但输入有 {x.shape[1]} 个 channel"
            )

        # BatchNorm2d 需要在 batch、height、width
        # 三个维度上计算统计量，保留 channel
        reduce_dims = (0, 2, 3)

        if self.training:
            # 每个 channel 至少需要两个数才能计算方差
            values_per_channel = x.numel() // x.shape[1]

            if values_per_channel <= 1:
                raise ValueError(
                    "训练 BatchNorm 时，"
                    "每个 channel 必须至少有两个数"
                )

            # 当前 batch 的均值
            # shape: (C,)
            mean = x.mean(
                dim=reduce_dims
            )

            # 当前 batch 的有偏方差
            # 用于当前这次标准化
            # shape: (C,)
            var = x.var(
                dim=reduce_dims,
                unbiased=False
            )

            # running statistics 不参与反向传播
            with torch.no_grad():
                # PyTorch 保存 running_var 时，
                # 使用无偏方差估计
                unbiased_var = x.var(
                    dim=reduce_dims,
                    unbiased=True
                )

                # running_mean 更新公式：
                # new = (1-momentum)*old
                #       + momentum*current
                self.running_mean.mul_(
                    1 - self.momentum
                )
                self.running_mean.add_(
                    self.momentum * mean
                )

                # running_var 使用相同更新方式
                self.running_var.mul_(
                    1 - self.momentum
                )
                self.running_var.add_(
                    self.momentum * unbiased_var
                )

        else:
            # inference 时不看当前 batch 的统计量
            # 直接使用训练过程中记录的历史统计量
            mean = self.running_mean
            var = self.running_var

        # 从 (C,) 变成 (1, C, 1, 1)，
        # 以便与 (N, C, H, W) 的 x 正确广播
        mean = mean.view(1, -1, 1, 1)
        var = var.view(1, -1, 1, 1)
        gamma = self.gamma.view(1, -1, 1, 1)
        beta = self.beta.view(1, -1, 1, 1)

        # 标准化
        x_hat = (x - mean) / torch.sqrt(
            var + self.eps
        )

        # 可学习的缩放和平移
        output = gamma * x_hat + beta

        return output

In [9]:
# Test Example: Batch Norm
torch.manual_seed(0)

bn = BatchNorm(num_channel=3)
bn.train()

x = torch.randn(4, 3, 8, 8)
y = bn(x)

print("输入形状：", x.shape)
print("输出形状：", y.shape)

output_mean = y.mean(
    dim=(0, 2, 3)
)

output_var = y.var(
    dim=(0, 2, 3),
    unbiased=False
)

print("\n输出每个 channel 的 mean：")
print(output_mean.detach())

print("\n输出每个 channel 的 var：")
print(output_var.detach())

print("\nrunning_mean：")
print(bn.running_mean)

print("\nrunning_var：")
print(bn.running_var)

输入形状： torch.Size([4, 3, 8, 8])
输出形状： torch.Size([4, 3, 8, 8])

输出每个 channel 的 mean：
tensor([-5.5879e-09, -7.4506e-09,  3.7253e-09])

输出每个 channel 的 var：
tensor([1.0000, 1.0000, 1.0000])

running_mean：
tensor([ 0.0132, -0.0016,  0.0072])

running_var：
tensor([1.0092, 1.0087, 0.9986])


In [7]:
# batch, sequence, dim
class LayerNorm(nn.Module):
    def __init__(self, num_channel, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(num_channel))
        self.beta = nn.Parameter(torch.zeros(num_channel))

    def forward(self, x):
        mean = x.mean(dim=-1, dimKeep=True)
        var = x.mean(dim=-1, dimKeep=True)
        x = (x-mean)/torch.sqrt(var+self.eps)
        return self.gamma*x+self.beta
